In [2]:
import polars as pl 
from dbconfig import engine
print("Environment ready!")

Environment ready!


In [7]:
pl.read_database(query = """
    select table_name,
    column_name, data_type
    from information_schema.columns
    where table_schema = 'raw'
    order by table_name, ordinal_position;""",
    connection = engine)

table_name,column_name,data_type
str,str,str
"""category_translation""","""product_category_name""","""text"""
"""category_translation""","""product_category_name_english""","""text"""
"""customers""","""customer_id""","""text"""
"""customers""","""customer_unique_id""","""text"""
"""customers""","""customer_zip_code_prefix""","""bigint"""
…,…,…
"""reviews""","""review_answer_timestamp""","""text"""
"""sellers""","""seller_id""","""text"""
"""sellers""","""seller_zip_code_prefix""","""bigint"""


In [9]:
customer_check = pl.read_database(query = """
    SELECT COUNT(*) AS rows,
    COUNT(DISTINCT customer_id) AS customer_ids,
    COUNT(DISTINCT customer_unique_id) AS unique_customers
    FROM raw.customers;""",
    connection = engine)
customer_check

rows,customer_ids,unique_customers
i64,i64,i64
99441,99441,96096


In [22]:
unique_check = pl.read_database(query = """
    SELECT customer_unique_id,
    COUNT(*) AS customer_records
    FROM raw.customers
    GROUP BY customer_unique_id
    HAVING COUNT(*) > 1
    ORDER BY customer_records DESC
    LIMIT 20;""",
    connection = engine)
unique_check

customer_unique_id,customer_records
str,i64
"""8d50f5eadf50201ccdcedfb9e2ac8455""",17
"""3e43e6105506432c953e165fb2acf44c""",9
"""1b6c7548a2a1f9037c1fd3ddfed95f33""",7
"""6469f99c1f9dfae7733b25662e7f1782""",7
"""ca77025e7201e3b30c44b472ff346268""",7
…,…
"""b4e4f24de1e8725b74e4a1f4975116ed""",5
"""56c8638e7c058b98aae6d74d2dd6ea23""",5
"""74cb1ad7e6d5674325c1f99b5ea30d82""",5


In [6]:
cusomter_to_order_ratio = pl.read_database(query = """
    SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(DISTINCT order_id) AS unique_orders
    FROM raw.orders;""",
    connection = engine)
cusomter_to_order_ratio

rows,unique_customers,unique_orders
i64,i64,i64
99441,99441,99441


In [7]:
order_check = pl.read_database(query = """
    SELECT COUNT(*) as rows,
    COUNT(DISTINCT order_id) as orders,
    COUNT(DISTINCT order_item_id) as order_items
    FROM raw.order_items;""",
    connection = engine)
order_check

rows,orders,order_items
i64,i64,i64
112650,98666,21


In [9]:
order_primary_check = pl.read_database(query = """
    SELECT COUNT(*) AS rows,
    COUNT(DISTINCT(order_id, order_item_id)) as composite_key
    FROM raw.order_items;""",
    connection = engine)
order_primary_check

rows,composite_key
i64,i64
112650,112650


In [21]:
maximum_basket = pl.read_database(query = """
    SELECT
    order_id,
    COUNT(*) AS item_count
    FROM raw.order_items
    GROUP BY order_id
    ORDER BY item_count DESC
    LIMIT 10;""",
    connection = engine)
maximum_basket

order_id,item_count
str,i64
"""8272b63d03f5f79c56e9e4120aec44ef""",21
"""1b15974a0141d54e36626dca3fdc731a""",20
"""ab14fdcfbe524636d65ee38360e22ce8""",20
"""428a2f660dc84138d969ccd69a0ab6d5""",15
"""9ef13efd6949e4573a18964dd1bbe7f5""",15
"""73c8ab38f07dc94389065f7eba4f297a""",14
"""9bdc4d4c71aa1de4606060929dee888c""",14
"""37ee401157a3a0b28c9c6d0ed8c3b24b""",13
"""3a213fcdfe7d98be74ea0dc05a8b31ae""",12


In [11]:
payment_check = pl.read_database(query = """
    SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT order_id) AS orders
    FROM raw.payments;""",
    connection = engine)
payment_check

rows,orders
i64,i64
103886,99440


In [12]:
payment_sequential_check = pl.read_database(query = """
    SELECT
    payment_sequential,
    COUNT(*) AS cnt
    FROM raw.payments
    GROUP BY payment_sequential
    ORDER BY payment_sequential;""",
    connection = engine)
payment_sequential_check

payment_sequential,cnt
i64,i64
1,99360
2,3039
3,581
4,278
5,170
…,…
25,2
26,2
27,1


In [13]:
payment_primary_key = pl.read_database(query = """SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT (order_id, payment_sequential)) AS composite_key
FROM raw.payments;""",
connection = engine)
payment_primary_key

rows,composite_key
i64,i64
103886,103886


In [14]:
review_check = pl.read_database(query = """
    SELECT COUNT(*) AS rows,
    COUNT(DISTINCT order_id) as orders,
    COUNT(DISTINCT review_id) AS reviews
    FROM raw.reviews;""",
    connection = engine)
review_check

rows,orders,reviews
i64,i64,i64
99224,98673,98410


In [15]:
null_check = pl.read_database(query = """SELECT
    COUNT(*) AS total_rows,
    COUNT(order_id) AS non_null_order_ids,
    COUNT(review_id) AS non_null_review_ids
    FROM raw.reviews;""",
    connection = engine)
null_check

total_rows,non_null_order_ids,non_null_review_ids
i64,i64,i64
99224,99224,99224


In [20]:
review_id_check = pl.read_database(query = """
SELECT review_id,
COUNT(*) AS cnt
FROM raw.reviews
GROUP BY review_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 20;""",
connection = engine)
review_id_check

review_id,cnt
str,i64
"""38821b5c496b678cf91acc34892805ad""",3
"""32415bbf6e341d5d517080a796f79b5c""",3
"""9e25d6e3025e9b9a0fc7f03588d33e2b""",3
"""39b4603793c1c7f5f36d809b4a218664""",3
"""2d6ac45f859465b5c185274a1c929637""",3
…,…
"""69a1068c3128a14994e3e422e4539e04""",3
"""3415c9f764e478409e8e0660ae816dd2""",3
"""0c76e7a547a531e7bf9f0b99cba071c1""",3


In [25]:
duplicate_check = pl.read_database(query = """SELECT
    review_id,
    order_id,
    review_score,
    review_comment_title
FROM raw.reviews
WHERE review_id = '38821b5c496b678cf91acc34892805ad';""", 
connection = engine)
duplicate_check

review_id,order_id,review_score,review_comment_title
str,str,i64,null
"""38821b5c496b678cf91acc34892805ad""","""2613fb342bec59126f9c5180a5bc95ec""",5,null
"""38821b5c496b678cf91acc34892805ad""","""02e723e8edb4a123d414f56cc9c4665e""",5,null
"""38821b5c496b678cf91acc34892805ad""","""2d4bc14df7f5eaf36d2ef6d9a5b7c0d8""",5,null


In [23]:
order_id_check = pl.read_database(query = """SELECT
order_id,
COUNT(*) AS cnt
FROM raw.reviews
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 20;""",
connection = engine)
order_id_check

order_id,cnt
str,i64
"""df56136b8031ecd28e200bb18e6ddb2e""",3
"""8e17072ec97ce29f0e1f111e598b0c85""",3
"""c88b1d1b157a9999ce368f218a407141""",3
"""03c939fd7fd3b38f8485a0f95798f1f6""",3
"""61b2314bd576f3d792e46e1d56e51ee6""",2
…,…
"""95287dc5df93f25b90309d5acddbe02e""",2
"""1041d437bb0a4bcfeb3a107426f2d5f5""",2
"""b7293e3014a7261f0d26d28a1e927864""",2


In [27]:
duplicate_order_id_check = pl.read_database(query = """
    SELECT
    review_id,
    order_id,
    review_score,
    review_comment_title
FROM raw.reviews
WHERE order_id = 'df56136b8031ecd28e200bb18e6ddb2e';""",
    connection = engine)
duplicate_order_id_check

review_id,order_id,review_score,review_comment_title
str,str,i64,null
"""c444278834184f72b1484dfe47de7f97""","""df56136b8031ecd28e200bb18e6ddb2e""",5,null
"""72a1098d5b410ae50fbc0509d26daeb9""","""df56136b8031ecd28e200bb18e6ddb2e""",5,null
"""44f3e54834d23c5570c1d010824d4d59""","""df56136b8031ecd28e200bb18e6ddb2e""",5,null


In [18]:
review_grain = pl.read_database(query = """SELECT
COUNT(*) AS rows,
COUNT(DISTINCT (review_id, order_id)) AS review_order_pairs
FROM raw.reviews;""",
connection = engine)
review_grain

rows,review_order_pairs
i64,i64
99224,99224


In [19]:
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [28]:
pl.read_database(\
query = """
SELECT *
FROM raw.reviews
WHERE order_id = 'df56136b8031ecd28e200bb18e6ddb2e';""",
connection = engine)

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
str,str,i64,null,null,str,str
"""c444278834184f72b1484dfe47de7f97""","""df56136b8031ecd28e200bb18e6ddb2e""",5,null,null,"""2017-02-08 00:00:00""","""2017-02-14 13:58:48"""
"""72a1098d5b410ae50fbc0509d26daeb9""","""df56136b8031ecd28e200bb18e6ddb2e""",5,null,null,"""2017-02-07 00:00:00""","""2017-02-10 10:46:09"""
"""44f3e54834d23c5570c1d010824d4d59""","""df56136b8031ecd28e200bb18e6ddb2e""",5,null,null,"""2017-02-09 00:00:00""","""2017-02-09 09:07:28"""


In [29]:
pl.read_database(
query = """
SELECT *
FROM raw.reviews
WHERE review_id = '38821b5c496b678cf91acc34892805ad';""",
connection = engine)

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
str,str,i64,null,null,str,str
"""38821b5c496b678cf91acc34892805ad""","""2613fb342bec59126f9c5180a5bc95ec""",5,null,null,"""2017-09-03 00:00:00""","""2017-09-05 12:12:51"""
"""38821b5c496b678cf91acc34892805ad""","""02e723e8edb4a123d414f56cc9c4665e""",5,null,null,"""2017-09-03 00:00:00""","""2017-09-05 12:12:51"""
"""38821b5c496b678cf91acc34892805ad""","""2d4bc14df7f5eaf36d2ef6d9a5b7c0d8""",5,null,null,"""2017-09-03 00:00:00""","""2017-09-05 12:12:51"""
